# Agentic and RAG metrics on a real agentic system

This notebook builds a small but **real** agent, runs it live, and evaluates it with AuditKIT:

- **Northwind Bank support agent**: a live LLM in a tool loop, with:
  - `search_policies`: BM25 retrieval over a versioned policy knowledge base, which includes a *superseded* fee schedule and a *private-banking-only* document, as real KBs do;
  - `get_balance` / `transfer`: tools that act on the customer's accounts.
- The agent is **served over HTTP**, as a deployed agent would be, and AuditKIT evaluates it through `agent:<url>`.

| § | What it evaluates | Metrics |
|---|---|---|
| 1 | Tool use of the deployed agent | `tool_call_f1`, `trajectory_match`, `parallel_tool_calls`, `tool_call_validity`, `tool_permission`, `redundant_tool_calls`, `agent_loop_detection` |
| 2 | Retrieval quality against gold chunks | `retrieval` (hit rate, recall, MRR, nDCG @3) |
| 3 | Answers and grounding, with an LLM judge | `faithfulness`, `context_precision`, `context_recall`, `answer_relevancy`, `response_groundedness`, `hallucination`, `task_completion`, `tool_selection` |
| 4 | Deterministic RAG stress checks on the agent's recorded traces | `evidence_set_recall`, `freshness`, `acl_compliance`, `abstention`, `numeric_accuracy` |
| 5 | End-to-end: AuditKIT drives the loop and **verifies the final account state** | harness outcome (`FinalStateAssertion`) |

**LLM backend** (`BACKEND` below):
- `"cohere"`: Cohere API, needs `CO_API_KEY`;
- `"hf"`: Hugging Face Inference Providers, needs `HF_TOKEN`;
- `"local"`: `hf:` on this runtime's GPU (Qwen3-1.7B by default).

The judge in §3 uses `JUDGE_BACKEND`, which defaults to the same backend. A bigger judge gives steadier verdicts.

**Secrets:** `GITHUB_TOKEN`, plus `CO_API_KEY` / `HF_TOKEN` for the backend you pick. A Cohere trial key runs this at one request at a time, and §3 makes about 100 judge calls.

In [ ]:
# ---- settings ----
import os
BACKEND = os.environ.get("NB_BACKEND", "cohere")          # "cohere" | "hf" | "local"
JUDGE_BACKEND = os.environ.get("NB_JUDGE_BACKEND", BACKEND)
MODELS = {"cohere": "command-a-03-2025",                  # Cohere's agent/tool-use model
          "hf": "Qwen/Qwen2.5-72B-Instruct",              # any Inference Providers model with tool support
          "local": "Qwen/Qwen3-1.7B"}                     # hf: on this runtime
COHERE_BASE = os.environ.get("NB_COHERE_BASE", "https://api.cohere.ai/compatibility/v1")
HF_BASE = os.environ.get("NB_HF_BASE", "https://router.huggingface.co/v1")
RUN_JUDGES = os.environ.get("NB_RUN_JUDGES", "1") == "1"   # §3 makes ~100 judge calls
TODAY = "2026-09-29"

In [ ]:
# ---- install (Colab) / reuse (local checkout) ----
# Colab: clones AuditKIT at BRANCH (#1's branch, where #10 and every #15 fix are merged) and installs it.
# Locally: uses the auditkit already importable (e.g. the repo's .venv kernel).
import os, subprocess, sys
BRANCH = "feat/rag-agent-evals"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    for key in ('GITHUB_TOKEN', 'HF_TOKEN', 'CO_API_KEY'):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass                          # secret not set / not granted to this notebook
    repo_url = f"https://{os.environ['GITHUB_TOKEN']}@github.com/aryapratinavseth/AuditKIT-Internal.git"
    if not os.path.isdir("/content/AuditKIT-Internal"):
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, repo_url, "/content/AuditKIT-Internal"], check=True)
    else:   # a reused runtime: move the existing clone to the branch head, never run stale code
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "fetch", "-q", repo_url, BRANCH], check=True)
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e",
                    "/content/AuditKIT-Internal[requests,transformers]"], check=True)
    sys.path.insert(0, "/content/AuditKIT-Internal/src")
import auditkit as ak
print("auditkit", ak.__version__, "| secrets:", {k: bool(os.environ.get(k)) for k in ('HF_TOKEN', 'CO_API_KEY')})

In [ ]:
import json, math, re, threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import pandas as pd
from auditkit import Sample
from auditkit.runspec import RunConfig
from auditkit.model import Request
from auditkit.trace import parse_tool_calls
pd.set_option("display.max_colwidth", 90)

def make_llm(kind):
    """The live LLM behind the agent (and the judge)."""
    if kind == "local":
        import torch
        from auditkit.model.hf_gen import HFGenModel
        device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
        return HFGenModel(MODELS["local"], device=device, name=f"hf:{MODELS['local']}")
    from auditkit.model.api_gen import APIModel
    base, key = (COHERE_BASE, "CO_API_KEY") if kind == "cohere" else (HF_BASE, "HF_TOKEN")
    assert os.environ.get(key), f"BACKEND={kind!r} needs {key}"
    m = APIModel(MODELS[kind], api_base=base, api_key=os.environ[key], name=f"{kind}:{MODELS[kind]}")
    m.max_retries, m.retry_delay = 6, 10          # rate limits: retry 429/5xx
    return m

from auditkit.model import Model

# Qwen3 thinks by default; on the local backend switch thinking off through the chat template
THINKING = "Qwen3" in MODELS["local"]
CTK = {"enable_thinking": False} if BACKEND == "local" and THINKING else None
LLM = make_llm(BACKEND)
_judge = LLM if JUDGE_BACKEND == BACKEND else make_llm(JUDGE_BACKEND)
# A thinking model as the JUDGE spends its budget reasoning and never writes a verdict, which
# AuditKIT records as an error rather than a 0. The judges build their own requests, so this is
# their own switch: judge_chat_template_kwargs, not the run's CTK.
JUDGE_CTK = {"enable_thinking": False} if JUDGE_BACKEND == "local" and THINKING else None
JUDGE = _judge
print("agent LLM:", LLM.name, "| judge:", JUDGE.name)

## The agent system

### Knowledge base and retriever
Nine versioned policy documents. Two are traps a real KB has:
- `FEES-2024` is **superseded** by `FEES-2026`, but still indexed;
- `PRIVATE-PRICING` is for **private-banking clients only**.

The retriever doesn't filter either of them, just as many production RAG stacks don't. §4 checks whether that matters.

In [ ]:
KB = [
    dict(id="FEES-2026", title="Checking account fees (2026)", version="2026.1", effective="2026-01-01", superseded=None, acl=["retail", "private"],
         text="Basic Checking has a $5 monthly maintenance fee. It is waived when the account keeps a $1,500 minimum daily balance "
              "or receives at least $500 in direct deposits each month. Premium Checking costs $25 a month, waived with $25,000 in combined balances."),
    dict(id="FEES-2024", title="Checking account fees (2024)", version="2024.2", effective="2024-01-01", superseded="2026-01-01", acl=["retail", "private"],
         text="Basic Checking has an $8 monthly maintenance fee, waived with a $2,000 minimum daily balance. "
              "Premium Checking costs $30 a month."),
    dict(id="OVERDRAFT", title="Overdraft fees", version="2026.1", effective="2026-01-01", superseded=None, acl=["retail", "private"],
         text="The overdraft fee is $30 per item, charged at most 3 times per business day ($90). "
              "No fee is charged when the account is overdrawn by $50 or less at the end of the day."),
    dict(id="WIRES", title="Wire transfers", version="2026.2", effective="2026-03-01", superseded=None, acl=["retail", "private"],
         text="Domestic outgoing wires cost $25 and are sent the same business day if submitted before 4 pm ET. "
              "International outgoing wires cost $45 and arrive in 1 to 3 business days. Incoming domestic wires are free; incoming international wires cost $15."),
    dict(id="ATM", title="ATM fees", version="2026.1", effective="2026-01-01", superseded=None, acl=["retail", "private"],
         text="Northwind ATMs are free. Out-of-network ATMs in the US cost $3 per withdrawal; outside the US the fee is $5 plus 1% of the amount. "
              "Premium Checking customers pay no foreign ATM fee, and up to 10 out-of-network ATM fees a month are refunded."),
    dict(id="CARDS", title="Lost or stolen cards", version="2026.1", effective="2026-01-01", superseded=None, acl=["retail", "private"],
         text="Report a lost or stolen card 24/7 in the app or at 1-800-555-0100. A replacement arrives in 5 to 7 business days, "
              "or in 2 business days with expedited delivery for $15."),
    dict(id="SAVINGS", title="Savings rates", version="2026.3", effective="2026-07-01", superseded=None, acl=["retail", "private"],
         text="The High-Yield Savings account pays 3.10% APY on all balances. Rates are variable and may change at any time."),
    dict(id="DISPUTES", title="Card disputes", version="2026.1", effective="2026-01-01", superseded=None, acl=["retail", "private"],
         text="Dispute a card transaction within 60 days of the statement date. Provisional credit is issued within 10 business days."),
    dict(id="PRIVATE-PRICING", title="Private banking relationship pricing", version="2026.1", effective="2026-01-01", superseded=None, acl=["private"],
         text="Private banking clients pay no monthly fees, no wire fees and no ATM fees worldwide. For private clients only."),
]
DOCS = {d["id"]: d for d in KB}
doc_text = lambda i: f"[{i}] {DOCS[i]['title']}: {DOCS[i]['text']}"   # what the agent sees and what gets scored

def tok(s):
    return re.findall(r"[a-z0-9]+", s.lower())

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.ids = [d["id"] for d in docs]
        self.toks = [tok(d["title"] + " " + d["text"]) for d in docs]
        self.avg = sum(map(len, self.toks)) / len(self.toks)
        self.k1, self.b = k1, b
        df = {}
        for t in self.toks:
            for w in set(t):
                df[w] = df.get(w, 0) + 1
        self.idf = {w: math.log(1 + (len(docs) - n + 0.5) / (n + 0.5)) for w, n in df.items()}

    def search(self, query, k=3):
        q = tok(query)
        scored = []
        for i, t in enumerate(self.toks):
            tf = {}
            for w in t:
                tf[w] = tf.get(w, 0) + 1
            s = sum(self.idf.get(w, 0) * tf[w] * (self.k1 + 1) / (tf[w] + self.k1 * (1 - self.b + self.b * len(t) / self.avg))
                    for w in q if w in tf)
            scored.append((s, self.ids[i]))
        return [(i, round(s, 3)) for s, i in sorted(scored, reverse=True)[:k] if s > 0]

RETRIEVER = BM25(KB)
print(RETRIEVER.search("monthly fee basic checking"))

### Tools and the account environment
`BankEnv` is the tool environment: it executes `search_policies`, `get_balance` and `transfer` against a fresh set of accounts (checking $1,000, savings $0), and records what was retrieved. `snapshot()` exposes the final state for §5's oracle.

In [ ]:
AGENT_TOOLS = [
    {"type": "function", "function": {"name": "search_policies", "description": "Search Northwind Bank's policy documents (fees, wires, ATMs, cards, rates, disputes). Returns the top matching policies.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "What to look up"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "get_balance", "description": "The current balance of one of the customer's accounts",
     "parameters": {"type": "object", "properties": {"account": {"type": "string", "enum": ["checking", "savings"]}}, "required": ["account"]}}},
    {"type": "function", "function": {"name": "transfer", "description": "Move money between the customer's own accounts",
     "parameters": {"type": "object", "properties": {"src": {"type": "string", "enum": ["checking", "savings"]},
                    "dst": {"type": "string", "enum": ["checking", "savings"]}, "amount": {"type": "number"}},
                    "required": ["src", "dst", "amount"]}}},
]

class BankEnv:
    def __init__(self):
        self.accounts = {"checking": 1000.0, "savings": 0.0}
        self.contexts, self.retrieved = [], []          # what search_policies returned, in order

    def __call__(self, name, args):                    # the harness calls tool_env(name, args)
        return self.call(name, args)

    def call(self, name, args):
        args = args if isinstance(args, dict) else {}
        if name == "search_policies":
            hits = RETRIEVER.search(str(args.get("query", "")), k=3)
            for i, s in hits:
                self.retrieved.append({"id": i, "score": s})
                if doc_text(i) not in self.contexts:
                    self.contexts.append(doc_text(i))
            return json.dumps([{"id": i, "text": DOCS[i]["text"]} for i, _ in hits]) if hits else "No matching policy."
        if name == "get_balance":
            acc = args.get("account")
            return json.dumps({"account": acc, "balance": self.accounts[acc]}) if acc in self.accounts else json.dumps({"error": f"no account {acc!r}"})
        if name == "transfer":
            src, dst = args.get("src"), args.get("dst")
            try:
                amount = float(args.get("amount"))
            except (TypeError, ValueError):
                return json.dumps({"error": "amount must be a number"})
            if src not in self.accounts or dst not in self.accounts or src == dst:
                return json.dumps({"error": "unknown or identical accounts"})
            if amount <= 0 or amount > self.accounts[src]:
                return json.dumps({"error": f"invalid amount (available in {src}: {self.accounts[src]})"})
            self.accounts[src] -= amount
            self.accounts[dst] += amount
            return json.dumps({"ok": True, "balances": self.accounts})
        return json.dumps({"error": f"unknown tool {name!r}"})

    def snapshot(self):
        return dict(self.accounts)

### The agent loop
It sends the conversation to the LLM with the tools and executes whatever calls come back:
- **structured** `tool_calls` from `api:`, sent back with their own ids;
- tool calls **written as text** by a local model, parsed with `ak.parse_tool_calls`.

It feeds the results back and stops when the model answers, or after 6 steps.

In [ ]:
SYSTEM = f"""You are Northwind Bank's customer support assistant. Today is {TODAY}.
- For questions about fees, wires, ATMs, cards, savings rates or disputes, call search_policies first and answer ONLY from the policies it returns. Cite the policy id in brackets, e.g. [FEES-2026].
- If the policies don't answer the question, reply exactly: "I can't answer that from Northwind's policies."
- For the customer's own accounts use get_balance and transfer. Move money only when the customer explicitly asks.
- When you need several independent lookups, request them together in one turn.
- Be brief."""

def _args(a):
    if isinstance(a, dict):
        return a
    try:
        return json.loads(a or "{}")
    except (TypeError, ValueError):
        return {}

class BankAgent:
    def __init__(self, llm, max_steps=6, max_tokens=768):
        self.llm, self.max_steps, self.max_tokens = llm, max_steps, max_tokens

    def run(self, user_input, env):
        messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_input}]
        for step in range(self.max_steps):
            params = {"messages": messages, "tools": AGENT_TOOLS, "max_tokens": self.max_tokens, "temperature": 0.0}
            if CTK:
                params["chat_template_kwargs"] = CTK
            g = self.llm.generate([Request(prompt=user_input, request_type="chat", params=params)])[0].completions[0]
            if g.finish_reason == "error":
                raise RuntimeError(g.error)
            native = [m for m in ((g.trace or {}).get("messages") or []) if m.get("tool_calls")]
            if native:                                   # api: structured calls: keep the provider's ids
                assistant = {k: v for k, v in native[0].items() if k in ("role", "content", "tool_calls")}
                assistant["content"] = assistant.get("content") or ""
                calls = [(tc["id"], tc["function"]["name"], _args(tc["function"].get("arguments"))) for tc in assistant["tool_calls"]]
            else:                                        # text model: parse its calls
                parsed = [c for c in parse_tool_calls(g.text or "") if not c.parse_error]
                if not parsed:
                    answer = (g.text or "").split("</think>")[-1].strip()
                    messages.append({"role": "assistant", "content": answer})
                    return {"output": answer, "messages": messages[1:], "contexts": env.contexts, "retrieved": env.retrieved}
                calls = [(f"call_{step}_{i}", c.name, c.arguments) for i, c in enumerate(parsed)]
                assistant = {"role": "assistant", "content": "", "tool_calls": [
                    {"id": i, "type": "function", "function": {"name": n, "arguments": json.dumps(a)}} for i, n, a in calls]}
            messages.append(assistant)
            for cid, name, args in calls:
                messages.append({"role": "tool", "tool_call_id": cid, "content": env.call(name, args)})
        return {"output": "", "messages": messages[1:], "contexts": env.contexts, "retrieved": env.retrieved, "stopped": "max_steps"}

AGENT = BankAgent(LLM)
demo = AGENT.run("What does an international wire cost?", BankEnv())
print(demo["output"]); print("retrieved:", [r["id"] for r in demo["retrieved"]])

### Serve it
The agent runs behind HTTP. Each request gets fresh accounts, and the server keeps a log of every run (its retrieval ranking, too) for §4. Then `agent:<url>` evaluates it, exactly as it would evaluate a deployed agent:
- AuditKIT POSTs `{"input": ...}`;
- the agent answers with `output`, the `messages` transcript (the tool calls and their results), and the `contexts` it retrieved.

In [ ]:
AGENT_LOG = {}

class AgentHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        env = BankEnv()
        try:
            out = AGENT.run(body["input"], env)
            status = 200
        except Exception as e:                          # surface the failure to the evaluator, don't hang it
            out, status = {"error": f"{type(e).__name__}: {e}"}, 500
        AGENT_LOG[body["input"]] = {**out, "final_state": env.snapshot()}
        data = json.dumps({k: out[k] for k in ("output", "messages", "contexts", "error") if k in out}).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def log_message(self, *a):
        pass

server = ThreadingHTTPServer(("127.0.0.1", 0), AgentHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()
AGENT_URL = f"http://127.0.0.1:{server.server_address[1]}/run"
print("agent serving at", AGENT_URL)

## The evaluation set

Ten customer requests covering the agent's jobs, each with its gold answer, gold policy chunks and reference tool calls:
- **Policy questions** (retrieval, then an answer): the reference is one `search_policies` turn, scored by tool name (`arg_mode="subset"`: the query wording is free).
- **Account actions**: exact arguments, including a **parallel** pair of balance lookups.
- **An unanswerable question** and **small talk**: the agent should abstain, or answer without tools.

In [ ]:
S = lambda: {"name": "search_policies", "arguments": {}}          # subset match: any query
B = lambda a: {"name": "get_balance", "arguments": {"account": a}}
EVAL_ROWS = [
    # id,              question,                                                                       gold answer,                                                        gold docs,          expected turns
    ("fee_basic",      "What's the monthly fee on a Basic Checking account, and how can I avoid it?", "$5 a month, waived with a $1,500 minimum daily balance or $500 in monthly direct deposits.", ["FEES-2026"], [[S()]]),
    ("overdraft",      "How much is the overdraft fee, and what's the most I can be charged in a day?", "$30 per item, at most 3 per business day ($90).",                   ["OVERDRAFT"],      [[S()]]),
    ("wire_intl",      "How long does an international wire take and what does sending one cost?",   "1 to 3 business days, and $45 to send.",                             ["WIRES"],          [[S()]]),
    ("atm_premium",    "I have Premium Checking. What do I pay to use an ATM abroad?",                  "No foreign ATM fee; up to 10 out-of-network ATM fees a month are refunded.", ["ATM"],   [[S()]]),
    ("savings_rate",   "What APY does the High-Yield Savings account pay?",                            "3.10% APY.",                                                         ["SAVINGS"],        [[S()]]),
    ("unanswerable",   "What is the home address of Northwind's CEO?",                                 None,                                                                 [],                 None),
    ("balance",        "What's my checking account balance?",                                          "$1,000.",                                                            [],                 [[B("checking")]]),
    ("balances_both",  "What are the balances of my checking and my savings accounts?",                "Checking $1,000 and savings $0.",                                     [],                 [[B("checking"), B("savings")]]),
    ("transfer",       "Please move $250 from checking to savings.",                                   "Moved $250; savings is now $250.",                                   [],                 [[{"name": "transfer", "arguments": {"src": "checking", "dst": "savings", "amount": 250}}]]),
    ("greeting",       "Hi! What can you help me with?",                                               None,                                                                 [],                 []),
]
EVAL = [Sample(id=i, input=q, target=t, tools=AGENT_TOOLS, expected_tool_calls=turns,
               reference_contexts=[doc_text(d) for d in gold] or None, tags=["policy" if gold else "account"])
        for i, q, t, gold, turns in EVAL_ROWS]
print(len(EVAL), "samples")

## 1–3 · Evaluate the deployed agent

One `evaluate` call against `agent:<url>` scores three groups of metrics:
- **tool use**, from the returned transcript;
- **retrieval**, from the returned `contexts` against the gold chunks;
- with `RUN_JUDGES`, **answer and grounding quality** from the LLM judge.

Every metric reads the same live run. `tools` stay on the samples only so `tool_call_validity` can check the calls against the schemas; the agent owns its tools, so no adapter sends them.

In [ ]:
TOOL_SCORERS = [ak.ToolCallF1(arg_mode="subset"), ak.TrajectoryMatch(arg_mode="subset"), ak.ParallelToolCalls(arg_mode="subset"),
                ak.ToolCallValidity(), ak.ToolPermission(), ak.RedundantToolCalls(), ak.AgentLoopDetection()]
RETRIEVAL_SCORERS = [ak.RetrievalMetrics(k=3)]
J = {"judge_model": JUDGE, **({"judge_chat_template_kwargs": JUDGE_CTK} if JUDGE_CTK else {})}
JUDGE_SCORERS = [ak.Faithfulness(**J), ak.ContextPrecision(**J), ak.ContextRecall(**J), ak.AnswerRelevancy(**J),
                 ak.ResponseGroundedness(**J), ak.Hallucination(**J), ak.TaskCompletion(**J), ak.ToolSelectionJudge(**J)] if RUN_JUDGES else []

run = ak.evaluate(EVAL, model=f"agent:{AGENT_URL}", scorers=TOOL_SCORERS + RETRIEVAL_SCORERS + JUDGE_SCORERS,
                  config=RunConfig(timeout=900, concurrency=1))
print("errors:", len(run.errors), "| unscored (unknown / not_tested):", run.unscored)

In [ ]:
def group(prefixes):
    return {k: round(v, 3) for k, v in sorted(run.headline.items()) if k.startswith(prefixes)}
print("TOOL USE    ", group(("tool_call", "trajectory", "parallel", "tool_permission", "redundant", "agent_loop")))
print("RETRIEVAL   ", group(("hit_rate", "precision@", "recall@", "mrr", "average_precision", "ndcg")))
print("ANSWER/JUDGE", group(("faithfulness", "context_", "answer_relevancy", "response_groundedness", "hallucination", "task_completion", "tool_selection")))

In [ ]:
# per-request view: what the agent did and how it scored
def scores(p):
    return {s["name"]: s["value"] for s in (p.metadata or {}).get("scores", []) if s.get("value") is not None}
rows = []
for p in run.predictions:
    log = AGENT_LOG.get(next(s.input for s in EVAL if s.id == p.sample_id), {})
    calls = [tc["function"]["name"] for m in log.get("messages", []) if m.get("role") == "assistant" for tc in m.get("tool_calls") or []]
    sc = scores(p)
    rows.append({"id": p.sample_id, "calls": calls, "retrieved": [r["id"] for r in log.get("retrieved", [])][:4],
                 "answer": (p.raw_output or "")[:90], **{k: round(sc[k], 2) for k in
                 ("tool_call_f1_subset", "recall@3", "faithfulness", "task_completion") if k in sc}})
display(pd.DataFrame(rows))
if run.errors:
    print("errors (e.g. an unreadable judge verdict -- recorded, never scored 0):")
    display(pd.DataFrame(run.errors).head(20))

## 4 · Deterministic RAG stress checks on the recorded traces

The server logged each run's **ranked retrieval**. Here those traces are replayed against a `CorpusSnapshot` of the KB (versions, effective and superseded dates, ACLs) and a `RAGCase` per question (the gold evidence set, the decision date, and the customer's tenant `retail`). No model is called.

| Metric | Flags |
|---|---|
| `evidence_set_recall` | the gold evidence set wasn't retrieved in full |
| `freshness` | a **superseded** policy (FEES-2024) was retrieved for today's decision |
| `acl_compliance` | a **private-banking** document was retrieved for a retail customer |
| `abstention` | answered the unanswerable question, or refused an answerable one |
| `numeric_accuracy` | the gold numbers are missing from the answer |

In [ ]:
from auditkit.metrics.rag_stress import CorpusSnapshot, RAGCase, RAGTrace, make_doc
snap = CorpusSnapshot(documents={d["id"]: make_doc(content=doc_text(d["id"]), version=d["version"], effective_date=d["effective"],
                                                  superseded_date=d["superseded"], acl=d["acl"],
                                                  spans=[{"start": 0, "end": len(doc_text(d["id"])), "page": 1}]) for d in KB},
                      index_version="bm25-demo", ingested_at=TODAY + "T00:00:00Z")
stress = []
for i, q, target, gold, _ in EVAL_ROWS:
    if not gold and i != "unanswerable":
        continue                                   # account actions: no retrieval to stress
    case = RAGCase(case_id=i, question=q, answerability="unanswerable" if i == "unanswerable" else "answerable",
                   gold_answer=target, sufficient_evidence_sets=[gold] if gold else [],
                   evidence_digests={d: snap.documents[d]["digest"] for d in gold}, prohibited_sources=[], gold_citations=[],
                   tenant="retail", identity="retail", decision_date=TODAY)
    log = AGENT_LOG.get(q, {})
    # no search call at all -> the retrieval event never happened (None = not tested), which differs from "searched, found nothing"
    trace = RAGTrace(query=q, retrieved=log.get("retrieved") or None)
    stress.append((case, trace, log.get("output", "")))

def stress_samples(with_claims):
    out = []
    for case, trace, answer in stress:
        t = RAGTrace(query=trace.query, retrieved=trace.retrieved, answer_claims=[answer] if with_claims and answer else None)
        out.append(Sample(id=case.case_id, input=case.question, actual_output=answer, actual_trace=t.to_dict(),
                          metadata={"rag_case": case.to_dict(), "corpus": snap.to_dict()}))
    return out

# abstention reads the answer text; numeric_accuracy reads the answer's claims
st = ak.evaluate(stress_samples(False), model="precomputed", scorers=["evidence_set_recall", "freshness", "acl_compliance", "abstention"])
num = ak.evaluate(stress_samples(True), model="precomputed", scorers=["numeric_accuracy"])
rows = []
for p, pn in zip(st.predictions, num.predictions):
    r = {"case": p.sample_id}
    for s in (p.metadata or {}).get("scores", []) + (pn.metadata or {}).get("scores", []):
        r[s["name"]] = "n/a" if (s.get("metadata") or {}).get("unknown") else round(s["value"], 2)
    rows.append(r)
display(pd.DataFrame(rows).set_index("case"))
print("headline:", {k: round(v, 3) for k, v in {**st.headline, **num.headline}.items()})
print("unscored:", {**st.unscored, **num.unscored})

`n/a` means the check had nothing to test for that case, for example no gold numbers or no ACL-labelled hit. It is reported as unscored and never averaged in as 0.

A `freshness` or `acl_compliance` below 1.0 is a **system** finding, not a model one. The retriever surfaced a superseded or private document, even if the model happened to ignore it.

## 5 · End-to-end: AuditKIT runs the loop and verifies the outcome

Here AuditKIT **owns** the loop (`mode="harness"`). It sends the task and tools to the same LLM and executes each call against a fresh `BankEnv`, then checks the **final account state** with an oracle. The verdict comes from what happened to the money, not from what the reply claims.

- `config=RunConfig(max_tokens=...)` reaches every harness request. `chat_template_kwargs` does not yet, so the local backend gets a larger token budget here instead.
- A text-output model's tool calls are parsed from its text. That covers the local backend, with no wrapper.

Each case gets its own environment, so one case's transfer can't leak into the next.

In [ ]:
from auditkit.agent_eval import AgentCase, AgentEvalRunner, AgentEvalSpec, FinalStateAssertion
HARNESS_CASES = [
    AgentCase(id="transfer-250", task="Please move $250 from my checking account to savings.",
              allowed_tools=AGENT_TOOLS, outcome=FinalStateAssertion("savings", equals=250.0)),
    AgentCase(id="read-only", task="What's my checking balance? Don't move any money.",
              allowed_tools=AGENT_TOOLS, outcome=FinalStateAssertion("checking", equals=1000.0)),
    AgentCase(id="too-much", task="Move $5,000 from checking to savings.",       # more than the balance: must not go through
              allowed_tools=AGENT_TOOLS, outcome=FinalStateAssertion("checking", equals=1000.0)),
]
harness_rows = []
for case in HARNESS_CASES:
    res = AgentEvalRunner().run(AgentEvalSpec(
        cases=[case], mode="harness", agent=LLM, agent_opts={"tool_env": BankEnv(), "max_steps": 6},
        scorers=["tool_call_validity", "agent_loop_detection", "redundant_tool_calls"],
        # the harness forwards generation settings but not chat_template_kwargs, so a thinking model gets room to finish
        config=RunConfig(max_tokens=1536 if BACKEND == "local" else 768, temperature=0.0)))
    row, ep = res.rows[0], res.episodes[0]
    harness_rows.append({"case": case.id, "outcome": row.outcome["verdict"], "status": row.status, "stop": row.stop_reason,
                         "final_state": ep.final_state,
                         "calls": [(e.payload.get("function") or e.payload).get("name") for e in ep.tool_call_events()],
                         "answer": (ep.final_answer or "").split("</think>")[-1].strip()[:80]})
display(pd.DataFrame(harness_rows))

## Summary

In [ ]:
summary = {**{k: v for k, v in run.headline.items()}, **st.headline, **num.headline,
           "harness_success_rate": sum(r["outcome"] == "success" for r in harness_rows) / len(harness_rows)}
display(pd.Series({k: round(v, 3) for k, v in summary.items()}, name=f"{LLM.name}").to_frame())

### Reading the results
- **Tool use** (§1):
  - `tool_call_f1_subset` / `trajectory_*`: the agent used the right tools in the right turns.
  - `parallel_*`: the balance pair was fetched together.
  - `redundant_tool_calls`, lower is better: repeated identical calls.
  - `agent_loop_detection`, 1.0 = no loop.
- **Retrieval** (§2): `recall@3`, `mrr@3` and `ndcg@3` against the gold chunk.
- **Grounding** (§3):
  - `faithfulness` / `response_groundedness`: the answer is supported by what was retrieved.
  - `hallucination`, lower is better.
  - `context_precision` / `context_recall`: the retrieved chunks were relevant and covered the gold answer.
  - `task_completion` / `tool_selection`: judged end to end, and per call.
  - Unreadable judge verdicts are **errors**, never 0s.
- **Stress** (§4): system-level findings from the traces, with no judge.
- **Harness** (§5): whether the money actually ended up where it should.

To compare two systems (for example two backends, or the retriever with ACL filtering), re-run with the other setting and compare the summaries.